In [1]:
import os
import glob
import numpy as np
from osgeo import gdal
from pathlib import Path

src_root = Path("/mnt/f/readyparams/modelprep")
dst_root = Path("/mnt/f/readyparams/rasters/current")
dst_root.mkdir(parents=True, exist_ok=True)

# 1. Iterate species folders (avoids slow recursive rglob)
for entry in os.scandir(src_root):
    if not entry.is_dir():
        continue
        
    species_name = entry.name
    species_path = entry.path
    
    # 2. Find the 4 seasonal files using a targeted glob
    # This matches anything ending in _spring.tif, _summer.tif, etc.
    seasonal_files = glob.glob(os.path.join(species_path, "*binary*.tif"))
    
    # Filter for the specific seasons to ensure we don't grab extra files
    seasons = ["spring", "summer", "fall", "winter"]
    target_files = [f for f in seasonal_files if any(s in f.lower() for s in seasons)]
    
    if len(target_files) == 0:
        continue

    # 3. Load and Stack in memory
    data_stack = []
    geo_transform = None
    projection = None
    
    for f_path in target_files:
        ds = gdal.Open(f_path)
        if geo_transform is None:
            geo_transform = ds.GetGeoTransform()
            projection = ds.GetProjection()
            cols, rows = ds.RasterXSize, ds.RasterYSize
            
        data_stack.append(ds.ReadAsArray())
        ds = None

    # 4. Compute Max (Vectorized numpy is near-instant for 12MB of data)
    max_composite = np.max(np.stack(data_stack), axis=0)

    # 5. Write final output
    out_path = dst_root / f"{species_name}.tif"
    driver = gdal.GetDriverByName("GTiff")
    
    # Using LZW + Predictor 2 + Tiling for best speed/size ratio
    out_ds = driver.Create(
        str(out_path), cols, rows, 1, gdal.GDT_Float32,
        options=["COMPRESS=LZW", "PREDICTOR=2", "TILED=YES", "NUM_THREADS=ALL_CPUS"]
    )
    out_ds.SetGeoTransform(geo_transform)
    out_ds.SetProjection(projection)
    
    band = out_ds.GetRasterBand(1)
    band.WriteArray(max_composite)
    band.SetNoDataValue(-9999)
    
    out_ds.FlushCache()
    out_ds = None
    print(f"✅ Created Max Composite: {species_name}")

print("\n✨ Done!")


/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


✅ Created Max Composite: anaxyrus_americanus
✅ Created Max Composite: anaxyrus_fowleri
✅ Created Max Composite: apalone_spinifera
✅ Created Max Composite: archilochus_colubris
✅ Created Max Composite: buteo_lineatus
✅ Created Max Composite: coccyzus_americanus
✅ Created Max Composite: dryocopus_pileatus
✅ Created Max Composite: elanoides_forficatus
✅ Created Max Composite: empidonax_virescens
✅ Created Max Composite: gastrophryne_carolinensis
✅ Created Max Composite: geothlypis_formosa
✅ Created Max Composite: hyla_avivoca
✅ Created Max Composite: hyla_chrysoscelis
✅ Created Max Composite: hyla_cinerea
✅ Created Max Composite: hyla_squirella
✅ Created Max Composite: hylocichla_mustelina
✅ Created Max Composite: kinosternon_subrubrum
✅ Created Max Composite: limnothlypis_swainsonii
✅ Created Max Composite: lithobates_catesbeianus
✅ Created Max Composite: lithobates_clamitans
✅ Created Max Composite: lithobates_sphenocephalus
✅ Created Max Composite: meleagris_gallopavo
✅ Created Max Com

In [2]:
from pathlib import Path
import shutil

src_root = Path("/mnt/f/readyparams/ppp_paramsoutput")
dst_root = Path("/mnt/f/readyparams/models")

dst_root.mkdir(parents=True, exist_ok=True)

for path in src_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".txt", ".pkl", ".csv", ".json"}:
        dst_path = dst_root / path.name  # ✅ flatten

        shutil.copy2(path, dst_path)
        print(f"Copied: {path} -> {dst_path}")

Copied: /mnt/f/readyparams/ppp_paramsoutput/anaxyrus_americanus/accuracy_tuned_anaxyrus_americanus.csv -> /mnt/f/readyparams/models/accuracy_tuned_anaxyrus_americanus.csv
Copied: /mnt/f/readyparams/ppp_paramsoutput/anaxyrus_americanus/best_beta_anaxyrus_americanus.txt -> /mnt/f/readyparams/models/best_beta_anaxyrus_americanus.txt
Copied: /mnt/f/readyparams/ppp_paramsoutput/anaxyrus_americanus/convex_hull_anaxyrus_americanus.json -> /mnt/f/readyparams/models/convex_hull_anaxyrus_americanus.json
Copied: /mnt/f/readyparams/ppp_paramsoutput/anaxyrus_americanus/elapid_maxent_model_tuned_anaxyrus_americanus.pkl -> /mnt/f/readyparams/models/elapid_maxent_model_tuned_anaxyrus_americanus.pkl
Copied: /mnt/f/readyparams/ppp_paramsoutput/anaxyrus_americanus/PermutationImportance_anaxyrus_americanus.csv -> /mnt/f/readyparams/models/PermutationImportance_anaxyrus_americanus.csv
Copied: /mnt/f/readyparams/ppp_paramsoutput/anaxyrus_americanus/PointNums_anaxyrus_americanus.txt -> /mnt/f/readyparams/mod